# Mobility network Data processing

After downloading all datasets (as described in raw/mobility/readme.txt) we proceed to process the data. 

We need to build the adjacency matrices for each snapshot.

In [1]:
import pandas as pd
import warnings
import xlrd
import numpy as np
import seaborn as sns
warnings.filterwarnings('ignore')

In [ ]:
input_folder = "../../data/raw/mobility/"
output_folder = "../../data/processed/mobility/"

!mkdir -p ../../data/processed/mobility/

### Adjacency matrices

We want to build the adjacency matrices. $A^t_{ij}$ corresponds to the population flow between county $i$ and $j$ at snapshot $t$. We store each snapshot independently in our output folder. 

We select only those counties that belong to the state of Texas (the state with the most counties). Their FIPS code starts with "48". We ensure that we have the same counties, and ordered in the same way, in each snapshot.

In [ ]:
list_matrices = []
list_counties = []

state_code = "48"  #Texas
weeks = ["2019_01_07", "2019_02_04", "2019_03_04", "2019_04_01", 
         "2019_05_06", "2019_06_03", "2019_07_01", "2019_08_05", 
         "2019_09_02", "2019_10_07", "2019_11_04", "2019_12_02"]

for week in weeks:
    print(f"week: {week}")
    file_name = f"../../data/mobility/weekly_county2county_{week}.csv"
    df = pd.read_csv(file_name, dtype=str)
    df = df[["geoid_o", "geoid_d", "pop_flows"]]
    df["pop_flows"] = df["pop_flows"].astype(float)
    
    #keep only counties in Texas
    df = df[(df['geoid_o'].str.startswith(state_code)) & (df['geoid_d'].str.startswith(state_code))]

    #adjacency matrix 
    unique_counties = np.unique(np.concatenate([df["geoid_o"].values, df["geoid_d"].values]))
    print(f"Number of counties = {len(unique_counties)}")
    dict_counties = {k: v for v, k in enumerate(unique_counties)}

    A = np.zeros((len(unique_counties), len(unique_counties)))
    for _, row in df.iterrows():
        i = dict_counties[row["geoid_o"]]
        j = dict_counties[row["geoid_d"]]
        A[i, j] = row["pop_flows"]

    list_matrices.append(A)
    list_counties.append(list(unique_counties))

    print("-------")

#find counties present in all snapshots:
set_counties_list = [set(sublist) for sublist in list_counties]
intersection_counties = set.intersection(*set_counties_list)
intersection_counties = sorted(intersection_counties)
print(f"Number of common counties (N) = {len(intersection_counties)}")

In [ ]:
#store adj matrices while ensuring the same order or nodes in each snapshot:
for w, week in enumerate(weeks):
    i_tokeep = [i for i in range(len(list_counties[w])) if list_counties[w][i] in intersection_counties]
    filtered_A = list_matrices[w][i_tokeep, :][:, i_tokeep]
    pd.DataFrame(filtered_A, index=intersection_counties, columns=intersection_counties).to_csv(f"{output_folder}/adjMat_{week}.csv")